# Generate Normalized Fiedler Values Across Layers

This Generate notebook computes and caches the per-layer normalized Fiedler values consumed by `Visualize-diagnostics-across-layers.ipynb`. The companion Visualize notebook compares layer-specific graph-spectral diagnostics across all 28 layers at a fixed context length of $N = 2000$ tokens for the main numerical input families using Llama-3.2-3B, providing the across-layer comparison presented under **Robustness and Layerwise Consistency of Graph-Spectral Diagnostics** in the appendix.

Set imports, constants, relative output paths, fixed-context configuration, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import random
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model, fixed-context sweep, and relative cache configuration
SEED = 42
MODEL_NAME = "meta-llama/Llama-3.2-3B"
EXPECTED_NUM_LAYERS = 28
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = CACHE_DIR / "fiedler_per_layer.npz"

CONTEXT_STATE_TOKENS = 1000
CONTEXT_COMMA_TOKENS = 1000
FIXED_CONTENT_TOKENS = 2000
TOTAL_SEQUENCE_STATES = 1001
NUM_ENSEMBLE_RUNS = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Define the sequence generators, tokenizer checks, and fixed-length ensemble used by the per-layer sweep.

In [ ]:
# Lorenz setup for the fixed 2,000-token context
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0

lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES, dtype=float) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))

In [ ]:
# Periodic and chaotic logistic-map regimes, quantization, and validation helpers
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_rs = list(periodic_logistic_periods.keys()) + [3.9, 4.0]
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(int(n), dtype=float)
    x[0] = float(x0)
    for i in range(1, int(n)):
        x[i] = float(r) * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    """Periodic logistic helper with optional post-burn-in phase offset."""
    if expected_period is None:
        vals = logistic_sequence(r, x0, n_window)
        return np.asarray(vals, dtype=float), 0, 0, False

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    vals = np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)
    return vals, int(t_settle), int(phase_offset), True


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


def _validate_numeric_sequence(sequence, data_name, rep_id):
    sequence = np.asarray(sequence, dtype=int)
    if sequence.shape != (TOTAL_SEQUENCE_STATES,):
        raise ValueError(
            f"{data_name}, rep_id={rep_id}: expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape}."
        )
    if sequence.min() < 0 or sequence.max() > 999:
        raise ValueError(f"{data_name}, rep_id={rep_id}: sequence values must be in [0, 999].")
    return sequence

In [ ]:
# Preserve a fixed input-family order in the saved cache
input_data_order = [
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
]

input_names = np.asarray(input_data_order, dtype=str)

In [ ]:
# Require each three-digit state and comma delimiter to map to one token
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def _single_token_id(token_str):
    token_ids = tokenizer(token_str, add_special_tokens=False).input_ids
    if len(token_ids) != 1:
        raise ValueError(f"Expected one token for {token_str!r}, got {token_ids}.")
    return int(token_ids[0])


COMMA_TOKEN_ID = _single_token_id(",")
NUMERIC_TOKEN_IDS = np.asarray([_single_token_id(f"{i:03d}") for i in range(1000)], dtype=np.int64)
if len(np.unique(NUMERIC_TOKEN_IDS)) != 1000:
    raise ValueError("Expected all 000..999 formatted states to map to distinct single tokens.")

In [ ]:
# Construct and validate the fixed post-BOS prompt
def _first_token_value(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return int(x.flatten()[0].item())
    return int(np.asarray(x).flatten()[0])


def get_token_start_index(
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
):
    if not drop_bos:
        return 0

    first_id = _first_token_value(input_ids)
    if first_id is not None and bos_token_id is not None and first_id == int(bos_token_id):
        return 1

    first_mask = _first_token_value(special_tokens_mask)
    if first_mask == 1:
        return 1

    return 0


def build_fixed_context_token_prompt(sequence):
    sequence = np.asarray(sequence, dtype=int)
    if sequence.size < CONTEXT_STATE_TOKENS:
        raise ValueError(
            f"Need at least {CONTEXT_STATE_TOKENS} state tokens; got {sequence.size}."
        )

    context_window = sequence[:CONTEXT_STATE_TOKENS]
    if context_window.min() < 0 or context_window.max() > 999:
        raise ValueError("Prompt state tokens must be in [0, 999].")

    prompt = ",".join([f"{x:03d}" for x in context_window]) + ","
    comma_count = prompt.count(",")
    if comma_count != CONTEXT_COMMA_TOKENS:
        raise ValueError(f"Expected {CONTEXT_COMMA_TOKENS} comma tokens, got {comma_count}.")

    content_token_count = int(context_window.size + comma_count)
    if content_token_count != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Expected {FIXED_CONTENT_TOKENS} content tokens, got {content_token_count}.")

    return prompt, int(context_window.size), int(comma_count), int(content_token_count)


def tokenize_and_validate_metric_prompt(prompt):
    encoded = tokenizer(
        prompt,
        add_special_tokens=True,
        return_tensors="pt",
    )
    input_ids = encoded.input_ids[0]
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=True,
    )
    post_bos_token_ids = input_ids[token_start:].detach().cpu().numpy().astype(np.int64)
    seq_len = int(post_bos_token_ids.shape[0])
    if seq_len != FIXED_CONTENT_TOKENS:
        raise ValueError(
            f"Expected post-BOS token count {FIXED_CONTENT_TOKENS}, got {seq_len} "
            f"(total_seq_len={int(input_ids.shape[-1])}, token_start={token_start})."
        )
    if int(post_bos_token_ids[-1]) != COMMA_TOKEN_ID:
        raise ValueError("Final post-BOS prompt token is not the comma token.")

    return {
        "encoded": encoded,
        "token_start": int(token_start),
        "seq_len": int(seq_len),
        "final_token_is_comma": True,
    }

In [ ]:
# Share each replicate's sampled initial condition across input families
ensemble_seed = SEED
periodic_random_phase = True
ensemble_rng = np.random.default_rng(ensemble_seed)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=NUM_ENSEMBLE_RUNS)
run_ids = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)

ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
ensemble_chaotic_rs = [3.9, 4.0]
ensemble_phase_rng = np.random.default_rng(ensemble_seed + 1)
ensemble_inputs = {}


def _store_entry(rep_inputs, rep_id, data_name, sequence):
    rep_inputs[data_name] = {
        "sequence": _validate_numeric_sequence(sequence, data_name=data_name, rep_id=rep_id),
    }


for rep_id, x0_rep in zip(run_ids, x0_samples):
    rep_id = int(rep_id)
    x0_rep = float(x0_rep)
    rep_inputs = {}

    constant_token = int(np.round(50 + 900 * x0_rep))
    _store_entry(
        rep_inputs,
        rep_id,
        "Constant sampled",
        np.full(TOTAL_SEQUENCE_STATES, constant_token, dtype=int),
    )

    for r in ensemble_periodic_rs:
        expected_period = logistic_expected_periods[r]
        if expected_period is None:
            raise ValueError(f"Expected a periodic window for r={r}, but expected_period is None.")

        phase_offset = (
            int(ensemble_phase_rng.integers(0, expected_period))
            if periodic_random_phase
            else 0
        )
        vals, _, _, _ = build_burned_in_logistic_window_ensemble(
            r,
            x0=x0_rep,
            n_window=TOTAL_SEQUENCE_STATES,
            expected_period=expected_period,
            n_long=logistic_long_n,
            tol=logistic_settle_tol,
            phase_offset=phase_offset,
        )
        _store_entry(rep_inputs, rep_id, f"Logistic r={r}", quantize_logistic_window(vals))

    for r in ensemble_chaotic_rs:
        vals = logistic_sequence(r, x0_rep, TOTAL_SEQUENCE_STATES)
        _store_entry(rep_inputs, rep_id, f"Logistic r={r}", quantize_logistic_window(vals))

    sol_rep = solve_ivp(
        lorenz,
        t_span,
        [float(-8.0 + (x0_rep - 0.5)), 7.0, 27.0],
        args=(sigma, rho, beta),
        t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape[-1] != TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Expected {TOTAL_SEQUENCE_STATES} Lorenz states for rep_id={rep_id}, "
            f"got {sol_rep.y.shape[-1]}."
        )
    _store_entry(rep_inputs, rep_id, "Lorenz", quantize_lorenz_x_values(sol_rep.y[0]))

    if list(rep_inputs.keys()) != input_data_order:
        raise ValueError(f"rep_id={rep_id}: unexpected input order {list(rep_inputs.keys())}.")
    ensemble_inputs[rep_id] = rep_inputs

if len(ensemble_inputs) != NUM_ENSEMBLE_RUNS:
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} ensemble runs, got {len(ensemble_inputs)}.")

Load the model with eager attention so the sweep can access full attention tensors.

In [ ]:
print("Loading model with eager attention implementation...")
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

fiedler_num_layers = int(model_attn.config.num_hidden_layers)
if fiedler_num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(
        f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {fiedler_num_layers}."
    )

layer_numbers = np.arange(1, fiedler_num_layers + 1, dtype=np.int64)
if not np.array_equal(layer_numbers, np.arange(1, EXPECTED_NUM_LAYERS + 1, dtype=np.int64)):
    raise ValueError("Expected saved layer numbers to be 1..28.")

print(
    "Model loaded: "
    f"{model_attn.config.num_hidden_layers} layers, "
    f"{model_attn.config.num_attention_heads} heads, "
    f"head_dim = {model_attn.config.hidden_size // model_attn.config.num_attention_heads}"
)

Compute the normalized Fiedler value from the normalized Laplacian of each layer's attention graph.

In [ ]:
# Token alignment, graph construction, and normalized Fiedler-value utilities
def _resolve_attention_layer_index(attentions, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(attentions) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(attentions):
        raise ValueError(
            f"Invalid attention layer_index={layer_index} for {len(attentions)} attention tensors"
        )
    return int(resolved_layer_index)


def compute_fiedler_from_attn(
    attentions,
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    head_weighting="mass",
    drop_bos=True,
    renormalize_after_drop=True,
    attention_layer_index=-1,
):
    """Compute the normalized Fiedler value for one attention layer."""
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=drop_bos,
    )
    resolved_attention_layer_index = _resolve_attention_layer_index(attentions, attention_layer_index)

    # Remove the leading BOS before constructing the token graph
    A = attentions[resolved_attention_layer_index][0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len <= 1:
        return {
            "fiedler_value": float("nan"),
            "seq_len": seq_len,
            "token_start": int(token_start),
            "resolved_attention_layer_index": int(resolved_attention_layer_index),
        }

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    # Symmetrize attention heads and combine them into one weighted graph
    W_h = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        alpha = s / s.sum()
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'")

    Wbar = (alpha[:, None, None] * W_h).sum(dim=0)

    # The second eigenvalue of the normalized Laplacian is the normalized Fiedler value
    deg = Wbar.sum(dim=1)
    inv_sqrt_deg = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    Wbar_norm = inv_sqrt_deg[:, None] * Wbar * inv_sqrt_deg[None, :]
    L_sym = torch.eye(Wbar.shape[0], device=Wbar.device, dtype=Wbar.dtype) - Wbar_norm

    eigvals = torch.linalg.eigvalsh(L_sym)
    fiedler_value = float("nan") if eigvals.numel() < 2 else eigvals[1].item()

    return {
        "fiedler_value": float(fiedler_value),
        "seq_len": int(seq_len),
        "token_start": int(token_start),
        "resolved_attention_layer_index": int(resolved_attention_layer_index),
    }


def compute_fiedler_per_layer_from_attn(
    attentions,
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    head_weighting="mass",
    drop_bos=True,
    renormalize_after_drop=True,
):
    return [
        compute_fiedler_from_attn(
            attentions,
            input_ids=input_ids,
            bos_token_id=bos_token_id,
            special_tokens_mask=special_tokens_mask,
            head_weighting=head_weighting,
            drop_bos=drop_bos,
            renormalize_after_drop=renormalize_after_drop,
            attention_layer_index=layer_idx,
        )
        for layer_idx in range(len(attentions))
    ]


fiedler_drop_bos = True
fiedler_renormalize_after_drop = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
fiedler_head_weighting = "mass"
fiedler_tokenization_add_special_tokens = True

print("Fiedler-value utilities ready.")
print("  BOS handling: default tokenizer special tokens are kept, then a leading BOS node is dropped.")
print("  Row renormalization after BOS removal: enabled.")
print("  Head symmetrization: W_h = 0.5 * (A_h + A_h.T).")
print(f"  Head aggregation: {fiedler_head_weighting}.")
print("  Laplacian: L_sym = I - D^(-1/2) Wbar D^(-1/2); metric is lambda_2.")

Run the full per-layer sweep, summarize across realizations, and save the visualization cache.

In [ ]:
device = next(model_attn.parameters()).device
expected_fiedler_shape = (len(input_names), len(run_ids), fiedler_num_layers)
if expected_fiedler_shape != (10, 20, 28):
    raise ValueError(f"Expected output shape (10, 20, 28), got {expected_fiedler_shape}.")

fiedler_per_run = np.full(expected_fiedler_shape, np.nan, dtype=np.float64)
# Task grid: realization x input family
tasks = [
    (int(rep_id), run_idx, data_name, input_idx)
    for run_idx, rep_id in enumerate(run_ids)
    for input_idx, data_name in enumerate(input_data_order)
]

for rep_id, run_idx, data_name, input_idx in tqdm(tasks, desc="Fixed-context Fiedler sweep"):
    entry = ensemble_inputs[int(rep_id)][data_name]
    sequence = np.asarray(entry["sequence"], dtype=int)

    prompt, state_token_count, comma_token_count, content_token_count = build_fixed_context_token_prompt(sequence)
    if state_token_count != CONTEXT_STATE_TOKENS:
        raise ValueError(f"Expected {CONTEXT_STATE_TOKENS} state tokens, got {state_token_count}.")
    if comma_token_count != CONTEXT_COMMA_TOKENS:
        raise ValueError(f"Expected {CONTEXT_COMMA_TOKENS} comma tokens, got {comma_token_count}.")
    if content_token_count != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Expected {FIXED_CONTENT_TOKENS} content tokens, got {content_token_count}.")

    prompt_info = tokenize_and_validate_metric_prompt(prompt)
    if prompt_info["seq_len"] != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Expected {FIXED_CONTENT_TOKENS} post-BOS tokens, got {prompt_info["seq_len"]}.")
    if not prompt_info["final_token_is_comma"]:
        raise ValueError("Every final prompt token must be the comma token.")

    inputs = prompt_info["encoded"].to(device)

    # Request attention tensors for all model layers
    with torch.no_grad():
        outputs = model_attn(
            **inputs,
            output_attentions=True,
            return_dict=True,
        )

    if outputs.attentions is None:
        raise ValueError("Model did not return attention weights.")
    if len(outputs.attentions) != fiedler_num_layers:
        raise ValueError(
            f"Expected {fiedler_num_layers} attention tensors, got {len(outputs.attentions)}."
        )

    # Compute the diagnostic for every layer using the same token alignment
    layer_bundles = compute_fiedler_per_layer_from_attn(
        outputs.attentions,
        input_ids=inputs.input_ids[0],
        bos_token_id=tokenizer.bos_token_id,
        head_weighting=fiedler_head_weighting,
        drop_bos=fiedler_drop_bos,
        renormalize_after_drop=fiedler_renormalize_after_drop,
    )
    if len(layer_bundles) != fiedler_num_layers:
        raise ValueError(f"Expected {fiedler_num_layers} layer metrics, got {len(layer_bundles)}.")

    for layer_idx, bundle in enumerate(layer_bundles):
        seq_len = int(bundle["seq_len"])
        resolved_layer_idx = int(bundle["resolved_attention_layer_index"])
        if seq_len != FIXED_CONTENT_TOKENS:
            raise ValueError(f"Layer {layer_idx + 1}: expected {FIXED_CONTENT_TOKENS} nodes, got {seq_len}.")
        if resolved_layer_idx != layer_idx:
            raise ValueError(
                f"Layer-order check failed: loop layer_idx={layer_idx}, resolved={resolved_layer_idx}."
            )
        fiedler_per_run[input_idx, run_idx, layer_idx] = float(bundle["fiedler_value"])

    del outputs, inputs, prompt_info
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if fiedler_per_run.shape != (10, 20, 28):
    raise ValueError(f"fiedler_per_run shape check failed: {fiedler_per_run.shape}")
if not np.array_equal(layer_numbers, np.arange(1, 29, dtype=np.int64)):
    raise ValueError(f"Expected layer_numbers == 1..28, got {layer_numbers}.")
if not np.isfinite(fiedler_per_run).all():
    bad_count = int(np.size(fiedler_per_run) - np.isfinite(fiedler_per_run).sum())
    raise ValueError(f"fiedler_per_run contains {bad_count} non-finite values.")

# Summarize across realizations
fiedler_mean = np.mean(fiedler_per_run, axis=1)
if len(run_ids) > 1:
    fiedler_std = np.std(fiedler_per_run, axis=1, ddof=1)
else:
    fiedler_std = np.zeros_like(fiedler_mean)

expected_summary_shape = (len(input_names), fiedler_num_layers)
for array_name, array_value in {
    "fiedler_mean": fiedler_mean,
    "fiedler_std": fiedler_std,
}.items():
    if array_value.shape != expected_summary_shape:
        raise ValueError(f"{array_name} has shape {array_value.shape}, expected {expected_summary_shape}.")

# Save compact arrays for the companion Visualize notebook
np.savez_compressed(
    OUTPUT_PATH,
    input_names=input_names,
    layer_numbers=layer_numbers,
    fiedler_mean=fiedler_mean,
    fiedler_std=fiedler_std,
)